# EXPERTA_MED — تدريب المصنّف واستنتاجه

تصنيف الجمل الطبية العربية (نساء وتوليد) إلى **21 صنفاً** باستخدام
`aubmindlab/bert-base-arabertv02`.

هذا الدفتر يغلّف السكربتات الموجودة في المشروع ولا يعيد كتابتها:

| المرحلة | ما يستدعيه الدفتر |
|---|---|
| التدريب | `train_arabert.py` (نفس الوسائط الموجودة في `build_arg_parser`) |
| الاستنتاج | `app.core.nlp.classifier.MedicalSentenceClassifier` |
| القراءة | `model_output/metrics.json` + `label_mapping.json` |

**ترتيب التشغيل:** ١ → ٢ → ٣ ثم إمّا ٤ (تدريب) أو ٦ (استنتاج بنموذج جاهز).

> التدريب يكتب فوق `model_output/best_model.pt`. إن كان لديك نسخة مدرّبة تريد
> الاحتفاظ بها، غيّر `output_dir` في القسم ٤ قبل التشغيل.

## ١. فحص البيئة

يحدّد ما إذا كان التدريب سيجري على GPU أم CPU — الفارق بين دقائق وساعات.

In [ ]:
import platform, sys

print("python   :", sys.version.split()[0], "|", platform.platform())

import torch
print("torch    :", torch.__version__)
print("cuda     :", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  gpu[{i}] : {p.name}  {p.total_memory / 1024**3:.1f} GB  sm_{p.major}{p.minor}")
    DEVICE = "cuda"
else:
    import os
    print("  cpu cores:", os.cpu_count())
    print("  ⚠️  لا يوجد GPU — التدريب سيكون بطيئاً جداً. قلّل epochs أو درّب على جهاز فيه CUDA.")
    DEVICE = "cpu"

import transformers, sklearn, numpy
print("transformers:", transformers.__version__)
print("sklearn     :", sklearn.__version__)
print("numpy       :", numpy.__version__)
print("\ndevice ->", DEVICE)

## ٢. جذر المشروع

كل السكربتات تفترض أن مجلد العمل هو جذر المستودع (تستخدم مسارات نسبية مثل
`data.jsonl` و `model_output/`)، لذا ننتقل إليه ونضيفه إلى `sys.path`.

In [ ]:
import os, sys
from pathlib import Path

# الدفتر داخل notebooks/ فالجذر هو الأب؛ يعمل أيضاً لو نُقل الدفتر إلى الجذر.
here = Path.cwd()
ROOT = here.parent if (here.parent / "train_arabert.py").exists() else here
assert (ROOT / "train_arabert.py").exists(), f"لم أجد train_arabert.py انطلاقاً من {here}"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_PATH  = ROOT / "data.jsonl"
MODEL_DIR  = ROOT / "model_output"

print("ROOT      :", ROOT)
print("data.jsonl:", DATA_PATH.exists(), f"({DATA_PATH.stat().st_size / 1024**2:.1f} MB)" if DATA_PATH.exists() else "")
print("model_output/:")
for f in sorted(MODEL_DIR.iterdir()) if MODEL_DIR.exists() else []:
    size = f.stat().st_size
    print(f"   {f.name:26} {size/1024**2:8.1f} MB" if f.is_file() else f"   {f.name:26} <dir>")

## ٣. استكشاف البيانات

`data.jsonl` سطر واحد لكل عيّنة: `{"text": ..., "label": ...}`.
التوزيع غير متوازن — لهذا يستخدم التدريب `class-weighted cross-entropy`.

In [ ]:
import json
from collections import Counter

rows = []
with open(DATA_PATH, encoding="utf-8") as fh:
    for i, line in enumerate(fh, 1):
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"  سطر {i} تالف: {e}")

print(f"عدد العيّنات: {len(rows):,}")

labels = Counter(r["label"] for r in rows)
lengths = [len(r["text"]) for r in rows]

print(f"عدد الأصناف : {len(labels)}")
print(f"طول النص    : وسيط={sorted(lengths)[len(lengths)//2]}  أقصى={max(lengths)}  متوسط={sum(lengths)/len(lengths):.0f} حرف")
print(f"             (max_len=128 توكن يغطي هذا التوزيع بسهولة)\n")

widest = max(len(k) for k in labels)
total = len(rows)
print(f"{'label':<{widest}}  {'count':>6}  {'share':>6}")
print("-" * (widest + 18))
for lab, n in labels.most_common():
    bar = "█" * max(1, round(40 * n / labels.most_common(1)[0][1]))
    print(f"{lab:<{widest}}  {n:>6}  {n/total:>5.1%}  {bar}")

imbalance = labels.most_common(1)[0][1] / min(labels.values())
print(f"\nنسبة عدم التوازن (أكبر/أصغر صنف): {imbalance:.1f}×")

In [ ]:
# عيّنات عشوائية للتحقق البصري من جودة الوسم
import random
random.seed(0)
for r in random.sample(rows, 8):
    print(f"[{r['label']:<20}] {r['text']}")

## ٤. التدريب

يستدعي `train_arabert.py` كعملية منفصلة ويعرض مخرجاتها لحظياً. تشغيله كعملية
مستقلة — لا `import` — يعني أن ذاكرة GPU تتحرّر كاملةً عند الانتهاء، ويمنع
تعارض حالة CUDA مع نواة الدفتر.

الوسائط أدناه هي نفس افتراضيات السكربت؛ عدّل ما تريد.

In [ ]:
# --- إعدادات التدريب (كلها اختيارية؛ القيم هنا هي افتراضيات السكربت) ---
TRAIN_ARGS = {
    "--data-path":       str(DATA_PATH),
    "--output-dir":      str(MODEL_DIR),
    "--model-name":      "aubmindlab/bert-base-arabertv02",
    "--max-len":         128,
    "--batch-size":      16,     # ارفعها إلى 32 على GPU بذاكرة ≥ 16GB
    "--epochs":          20,
    "--lr":              2e-5,
    "--warmup-ratio":    0.1,
    "--dropout":         0.3,
    "--label-smoothing": 0.1,
    "--llrd-decay":      0.95,   # layer-wise LR decay؛ 1.0 = معدّل موحّد
    "--patience":        5,      # early stopping على val macro-F1
    "--num-workers":     2,      # 0 على ويندوز/WSL إن ظهرت مشاكل
    "--seed":            42,
}

argv = []
for k, v in TRAIN_ARGS.items():
    argv += [k, str(v)]
print("سيُنفَّذ:\n  python train_arabert.py \\\n    " + " \\\n    ".join(
    f"{k} {v}" for k, v in TRAIN_ARGS.items()))

In [ ]:
import subprocess, sys, time

t0 = time.time()
proc = subprocess.Popen(
    [sys.executable, "train_arabert.py", *argv],
    cwd=str(ROOT),
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, encoding="utf-8", errors="replace",
)
for line in proc.stdout:
    print(line, end="")
rc = proc.wait()

mins = (time.time() - t0) / 60
print(f"\n{'='*60}\nانتهى التدريب: rc={rc}  المدة={mins:.1f} دقيقة")
if rc != 0:
    print("⚠️  فشل التدريب — راجع آخر سطور المخرجات أعلاه.")

## ٥. نتائج التدريب

`metrics.json` يكتبه السكربت في نهاية التدريب: F1 على مجموعة الاختبار،
تقرير لكل صنف، مصفوفة الالتباس، ومنحنى التدريب.

In [ ]:
import json

metrics_path = MODEL_DIR / "metrics.json"
if not metrics_path.exists():
    print("لا يوجد metrics.json بعد — شغّل القسم ٤ أولاً.")
else:
    M = json.loads(metrics_path.read_text(encoding="utf-8"))

    print(f"أفضل epoch          : {M['best_epoch']}")
    print(f"أفضل val macro-F1   : {M['best_val_macro_f1']:.4f}")
    print(f"test macro-F1       : {M['test']['macro_f1']:.4f}")
    print(f"test weighted-F1    : {M['test']['weighted_f1']:.4f}")
    print(f"تقسيم البيانات      : {M['data']['splits']}")

    per_class = M["test"]["per_class"]
    print(f"\n{'label':<22} {'prec':>6} {'recall':>7} {'f1':>6} {'n':>5}")
    print("-" * 50)
    rows_ = [(k, v) for k, v in per_class.items() if isinstance(v, dict) and "f1-score" in v]
    for k, v in sorted(rows_, key=lambda kv: kv[1]["f1-score"]):
        flag = "  ← أضعف" if v["f1-score"] < 0.70 else ""
        print(f"{k:<22} {v['precision']:>6.2f} {v['recall']:>7.2f} {v['f1-score']:>6.2f} {int(v['support']):>5}{flag}")

In [ ]:
# منحنى التدريب — يكشف overfitting (تحسّن train مع ثبات/تراجع val)
if metrics_path.exists() and M.get("history"):
    H = M["history"]
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
        ep = [h["epoch"] for h in H]
        ax[0].plot(ep, [h["train_loss"] for h in H], marker="o", label="train loss")
        if "val_loss" in H[0]:
            ax[0].plot(ep, [h["val_loss"] for h in H], marker="o", label="val loss")
        ax[0].set_xlabel("epoch"); ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=.3)
        ax[1].plot(ep, [h["val_macro_f1"] for h in H], marker="o", color="tab:green", label="val macro-F1")
        ax[1].axvline(M["best_epoch"], ls="--", c="grey", label=f"best (ep {M['best_epoch']})")
        ax[1].set_xlabel("epoch"); ax[1].set_title("Validation macro-F1"); ax[1].legend(); ax[1].grid(alpha=.3)
        plt.tight_layout(); plt.show()
    except ImportError:
        print("matplotlib غير مثبّت — عرض نصّي:\n")
        for h in H:
            print(f"  ep {h['epoch']:>2}  train_loss={h['train_loss']:.4f}  val_macro_f1={h['val_macro_f1']:.4f}")

In [ ]:
# مصفوفة الالتباس — أين يخلط النموذج بين الأصناف
if metrics_path.exists():
    import numpy as np
    cm = np.array(M["test"]["confusion_matrix"])
    names = M["test"]["labels"]

    try:
        import matplotlib.pyplot as plt
        cmn = cm / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
        fig, ax = plt.subplots(figsize=(10, 8.5))
        im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=8)
        ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=8)
        ax.set_xlabel("predicted"); ax.set_ylabel("true")
        ax.set_title("Confusion matrix (row-normalised)")
        fig.colorbar(im, fraction=0.046)
        plt.tight_layout(); plt.show()
    except ImportError:
        print("matplotlib غير مثبّت — تُعرض أسوأ الالتباسات فقط.\n")

    # أكبر الأخطاء خارج القطر، أياً كانت طريقة العرض
    off = [(names[i], names[j], int(cm[i][j]))
           for i in range(len(names)) for j in range(len(names))
           if i != j and cm[i][j] > 0]
    print("أكثر 12 التباساً:")
    for t, p, n in sorted(off, key=lambda x: -x[2])[:12]:
        print(f"  {n:>3}×   {t}  →  {p}")

## ٦. الاستنتاج

`MedicalSentenceClassifier` يقرأ `model_config.json` و`label_mapping.json`
ويتحقّق من تطابقهما مع أوزان `best_model.pt` قبل الاستخدام — أي عدم تطابق
يوقف التحميل بدل أن يعطي تصنيفات خاطئة بصمت.

> إن كانت هذه أوّل مرة، سيُنزّل نموذج AraBERT الأساس من HuggingFace
> (~540MB) ما لم يكن `model_output/bert/` موجوداً.

In [ ]:
from app.core.nlp.classifier import MedicalSentenceClassifier
from app.core.nlp.sections import label_ar, soap_for_label

clf = MedicalSentenceClassifier(model_dir=str(MODEL_DIR))
print("جاهز.")
print("  device      :", clf.device)
print("  max_len     :", clf.max_len)
print("  عدد الأصناف :", clf.num_classes)
print("  temperature :", clf.temperature, "(1.0 = بدون معايرة)")

In [ ]:
SENTENCES = [
    "حاسة بحركة الجنين خفت كتير اليوم وصارلها كم ساعة ما تحركت",
    "بتتحسس من البنسلين وبيعملها طفح جلدي وحكة وضيق نفس شديد",
    "تحليل HbA1c طلع مرتفع بشكل واضح يدل على سوء ضبط السكري",
    "نوصي بإيقاف الميتفورمين مؤقتاً ومراقبة السكر يومياً",
    "المريضة في الأسبوع 34 مع علامات تسمم حمل وارتفاع ضغط",
    "الضغط 130/85 والنبض 78 والحرارة 37.1",
    "موعد المراجعة بعد أسبوعين مع إعادة تحليل الدم",
]

results = clf.predict_batch(SENTENCES)

print(f"{'label':<20} {'conf':>6}  {'عربي':<22} {'SOAP':<10}")
print("-" * 78)
for text, (label, conf) in zip(SENTENCES, results):
    low = " ⚠️" if conf < clf.low_confidence_threshold else ""
    print(f"{label:<20} {conf:>5.1%}  {label_ar(label):<22} {soap_for_label(label):<10}{low}")
    print(f"    {text}\n")

In [ ]:
# احتمالات كل الأصناف لجملة واحدة — مفيد لفهم الحالات الحديّة
text = "المريضة تشكو من ألم أسفل البطن مع نزيف خفيف منذ يومين"
label, conf, probs = clf.predict(text, return_probs=True)

print(text)
print(f"\nالتصنيف: {label} ({conf:.1%})  →  {label_ar(label)} / {soap_for_label(label)}\n")
print("أعلى 6 احتمالات:")
for lab, p in sorted(probs.items(), key=lambda kv: -kv[1])[:6]:
    bar = "█" * round(p * 40)
    print(f"  {lab:<20} {p:>6.1%}  {bar}")

### ٦.١ الاستنتاج مع قياس الثقة

`predict_with_uncertainty` يضيف ثلاث إشارات لا يعطيها softmax وحده:
**الإنتروبيا** (تشوّش بين أصناف معروفة)، **تباين MC-dropout** (هل يصمد الجواب
عند إعادة أخذ العيّنة)، و**Mahalanobis OOD** (هل تشبه الجملة بيانات التدريب أصلاً).

يعمل فقط إن كان `model_output/train_stats.json` موجوداً (يكتبه التدريب).

In [ ]:
if clf.train_stats is None:
    print("train_stats.json غير موجود — قياس OOD معطّل لهذه النسخة من النموذج.")
else:
    probe = [
        "المريضة في الأسبوع 34 مع علامات تسمم حمل",
        "الطقس اليوم جميل وذهبت إلى السوق لشراء الخضار",   # خارج التوزيع عمداً
    ]
    for text, r in zip(probe, clf.predict_with_uncertainty(probe)):
        u = r["uncertainty"]
        print(f"{text}\n  → {r['label']} ({r['confidence']:.1%})")
        print(f"    entropy={u.entropy:.3f}  mc_variance={u.mc_variance:.4f}  ood={u.ood_score:.2f}\n")

## ٧. بعد ذلك

- **جملك أنت:** استبدل `SENTENCES` أعلاه وأعد تشغيل الخلية — لا حاجة لإعادة التحميل.
- **الخدمة الكاملة** (صوت → تقرير) ليست في هذا الدفتر؛ شغّلها من الطرفية:
  ```
  uvicorn app.main:app --host 0.0.0.0 --port 8000
  ```
  ثم افتح `http://<tailscale-ip>:8000/docs`. تحتاج `ffmpeg` مثبّتاً لأجل Whisper.
- **إعادة التدريب على بيانات جديدة:** أضف أسطراً إلى `data.jsonl` وأعد القسم ٤،
  أو مرّر ملفاً إضافياً عبر `--extra-data path.jsonl --extra-weight 3.0`.
- **الاختبارات:** `pytest -q` من جذر المشروع.